# US-3: Validate Crawl-Only Approach on Detail Pages

This notebook validates that crawling detail URLs directly produces high-quality data.

**Validation Goals:**
1. ✅ Prove crawl jobs can be submitted successfully
2. ✅ Confirm RSC extraction works for detail pages
3. ✅ Verify data quality (field completeness, success rate)
4. ✅ Estimate cost and time for full production run
5. ✅ Document any issues or edge cases

**Scope:** Test with available sample detail URLs (~20)
**Time:** ~30 minutes
**Cost:** ~$0.05 (negligible)

## 1. Setup & Imports

In [1]:
import httpx
import polars as pl
import json
import re
import os
import time
from datetime import datetime
from tqdm import tqdm

# Load environment
with open(".env") as f:
    for line in f:
        if "=" in line:
            key, value = line.strip().split("=", 1)
            os.environ[key] = value

ACCOUNT_ID = os.environ["CLOUDFLARE_ACCOUNT_ID"]
API_TOKEN = os.environ["CLOUDFLARE_API_TOKEN"]
BASE_URL = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/browser-rendering"

print(f"Validation started at: {datetime.now().isoformat()}")

Validation started at: 2026-03-22T19:33:51.665417


## 2. API Helper Functions

In [2]:
def get_headers():
    """Return authorization headers for Cloudflare API."""
    return {
        "Authorization": f"Bearer {API_TOKEN}",
        "Content-Type": "application/json"
    }


def submit_single_page_crawl(url: str) -> str:
    """Submit a single-page crawl job (depth=1, limit=1).
    
    Returns the job ID as a string.
    """
    payload = {
        "url": url,
        "limit": 1,
        "depth": 1,
        "source": "links",
        "formats": ["html"],
        "render": True,
        "options": {
            "includeExternalLinks": False,
            "includeSubdomains": False
        },
        "gotoOptions": {
            "waitUntil": "networkidle2",
            "timeout": 60000
        },
        "rejectResourceTypes": ["image", "media", "font", "stylesheet"]
    }
    
    with httpx.Client(timeout=120.0) as client:
        response = client.post(f"{BASE_URL}/crawl", headers=get_headers(), json=payload)
        result = response.json()
    
    if not result.get("success"):
        raise Exception(f"Crawl failed: {result}")
    
    # The API returns result as a string (job ID directly), not a dict
    job_id = result.get("result")
    if isinstance(job_id, str):
        return job_id
    elif isinstance(job_id, dict):
        return job_id.get("id")
    else:
        raise Exception(f"Unexpected result type: {type(job_id)}, content: {result}")


def poll_crawl(job_id: str, timeout: int = 300) -> dict:
    """Poll crawl job until completion."""
    start = time.time()
    while time.time() - start < timeout:
        with httpx.Client(timeout=60.0) as client:
            response = client.get(
                f"{BASE_URL}/crawl/{job_id}",
                headers=get_headers(),
                params={"limit": 100}
            )
            result = response.json()
        
        if not result.get("success"):
            raise Exception(f"Poll failed: {result}")
        
        data = result.get("result", {})
        status = data.get("status")
        
        if status == "completed":
            return {
                "status": "completed",
                "records": data.get("records", []),
                "browser_seconds": data.get("browserSecondsUsed", 0)
            }
        elif status in ["cancelled", "errored"]:
            return {"status": status, "records": [], "error": data.get("error")}
        
        time.sleep(5)
    
    raise TimeoutError(f"Job {job_id} timed out after {timeout}s")


print("API functions defined.")

API functions defined.


## 3. Load Sample Detail URLs (AC1)

In [ ]:
print("=== AC1: Load Sample Detail URLs ===\n")

# Load detail URLs
with open("output/detail_urls.json") as f:
    all_urls = json.load(f)

# Normalize to absolute URLs
normalized_urls = []
for url in all_urls:
    if url.startswith('/'):
        url = f"https://www.sgcarmart.com{url}"
    normalized_urls.append(url)

# Use all available URLs (we have ~20)
sample_urls = normalized_urls

# Store total count for cost estimation (derived from actual data)
TOTAL_DETAIL_URLS_AVAILABLE = len(all_urls)

print(f"Total detail URLs available: {len(all_urls)}")
print(f"Sample size for validation: {len(sample_urls)}")
print(f"\nSample URLs:")
for url in sample_urls[:3]:
    print(f"  {url}")
print(f"  ... ({len(sample_urls) - 3} more)")

# Save sample for reference
with open("output/validation_sample_urls.json", 'w') as f:
    json.dump(sample_urls, f, indent=2)

print(f"\n✅ Saved sample URLs to output/validation_sample_urls.json")

## 4. Submit Crawl Jobs (AC2)

Submit single-page crawl jobs for each detail URL.

In [4]:
print("\n=== AC2: Submit Crawl Jobs ===\n")

job_mapping = {}

for url in tqdm(sample_urls, desc="Submitting jobs"):
    try:
        job_id = submit_single_page_crawl(url)
        job_mapping[url] = {
            "job_id": job_id,
            "status": "submitted"
        }
    except Exception as e:
        job_mapping[url] = {
            "status": "submission_error",
            "error": str(e)
        }
    time.sleep(0.5)  # Small delay to avoid rate limiting

submitted = sum(1 for v in job_mapping.values() if v.get("job_id"))
errors = sum(1 for v in job_mapping.values() if v.get("status") == "submission_error")

print(f"\n✅ Submitted: {submitted}/{len(sample_urls)}")
print(f"❌ Errors: {errors}")

# Save checkpoint
with open("output/validation_jobs.json", 'w') as f:
    json.dump(job_mapping, f, indent=2)

print(f"\n✅ Saved job mapping to output/validation_jobs.json")


=== AC2: Submit Crawl Jobs ===



Submitting jobs: 100%|██████████| 20/20 [00:41<00:00,  2.07s/it]


✅ Submitted: 20/20
❌ Errors: 0

✅ Saved job mapping to output/validation_jobs.json


## 5. Poll and Collect Results (AC3)

Poll each job and collect HTML results.

In [5]:
print("\n=== AC3: Poll and Collect Results ===\n")

total_browser_seconds = 0
completed_count = 0

for url, info in tqdm(job_mapping.items(), desc="Polling jobs"):
    if info.get("status") != "submitted":
        continue
    
    job_id = info["job_id"]
    
    try:
        result = poll_crawl(job_id, timeout=300)
        
        if result["status"] == "completed":
            records = result.get("records", [])
            html = records[0].get("html", "") if records else ""
            
            job_mapping[url].update({
                "status": "completed",
                "html": html,
                "browser_seconds": result.get("browser_seconds", 0)
            })
            completed_count += 1
            total_browser_seconds += result.get("browser_seconds", 0)
        else:
            job_mapping[url].update({
                "status": result["status"],
                "error": result.get("error")
            })
    except Exception as e:
        job_mapping[url].update({
            "status": "poll_error",
            "error": str(e)
        })

# Update checkpoint
with open("output/validation_jobs.json", 'w') as f:
    json.dump(job_mapping, f, indent=2)

print(f"\n✅ Completed: {completed_count}/{len(sample_urls)}")
print(f"Success rate: {(completed_count/len(sample_urls)*100):.1f}%")
print(f"Total browser time: {total_browser_seconds:.2f}s")


=== AC3: Poll and Collect Results ===



Polling jobs: 100%|██████████| 20/20 [00:22<00:00,  1.11s/it]


✅ Completed: 20/20
Success rate: 100.0%
Total browser time: 11.77s


## 6. Extract Data from RSC Payload (AC4)

Extract detail fields from the `self.__next_f.push()` RSC payload.

In [ ]:
print("\n=== AC4: Extract Data from RSC Payload ===\n")

# First, let's inspect what's in the HTML
debug_html = None
debug_url = None
for url, info in job_mapping.items():
    if info.get("status") == "completed" and info.get("html"):
        debug_html = info["html"]
        debug_url = url
        break

if debug_html:
    print(f"Inspecting HTML from: {debug_url}")
    print(f"HTML length: {len(debug_html):,} characters\n")
    
    # Check for various markers
    markers = [
        ("self.__next_f", "self.__next_f"),
        ("ucInfoDetailData", "ucInfoDetailData"),
        ("ucInfoDetailTopData", "ucInfoDetailTopData"),
    ]
    
    for name, marker in markers:
        count = debug_html.count(marker)
        print(f"  {name}: {count} occurrences")


def parse_dollar_value(value):
    """Parse dollar-formatted values like '$$15,570 /yr' to numeric.
    
    Returns None if parsing fails (instead of returning the raw string).
    """
    if value is None:
        return None
    # Remove $ and commas, extract numeric part
    cleaned = re.sub(r'[$,]', '', str(value))
    # Extract first number (before any /)
    match = re.search(r'([\d]+)', cleaned)
    if match:
        return int(match.group(1))
    return None  # Return None instead of raw value to avoid mixed types


def extract_all_rsc_content(html: str) -> str:
    """Extract and combine all RSC push content from HTML."""
    push_pattern = r'self\.__next_f\.push\(\[(\d+),\s*"(.+?)"\]\)'
    push_matches = re.findall(push_pattern, html, re.DOTALL)
    
    all_content = ""
    for index_str, content in push_matches:
        try:
            unescaped = content.encode().decode('unicode_escape')
            all_content += unescaped
        except:
            all_content += content.replace('\\"', '"').replace('\\n', '\n')
    
    return all_content


def find_data_object(content: str, start_pos: int) -> dict | None:
    """Find and parse a data object starting from a position."""
    data_start = content.find('"data":{', start_pos)
    if data_start < 0:
        return None
    
    # Check if this is actually an object (not a string like "data":"$159")
    next_chars = content[data_start+7:data_start+20]
    if '{' not in next_chars[:3]:
        return None  # data is not an object
    
    brace_count = 0
    start_idx = data_start + 7
    i = start_idx
    while i < len(content):
        if content[i] == '{':
            brace_count += 1
        elif content[i] == '}':
            brace_count -= 1
            if brace_count == 0:
                break
        i += 1
    
    json_str = content[start_idx:i+1]
    
    try:
        return json.loads(json_str)
    except:
        return None


def extract_detail_from_html(html: str, url: str) -> dict:
    """Extract car detail fields from HTML.
    
    Numeric fields that fail to parse will be None (not mixed types).
    """
    detail_data = {"detail_page_url": url}
    
    # Extract all RSC content
    all_content = extract_all_rsc_content(html)
    
    # Step 1: Find ucInfoDetailData - search for ALL occurrences
    # because some listings have multiple and the first one might not have the detail fields
    search_pos = 0
    detail_data_found = None
    
    while True:
        detail_pos = all_content.find('"ucInfoDetailData"', search_pos)
        if detail_pos < 0:
            break
        
        # Check for success:true
        success_pos = all_content.find('"success":true', detail_pos)
        if success_pos > 0 and success_pos < detail_pos + 100:
            data_obj = find_data_object(all_content, success_pos)
            if data_obj and 'omv' in data_obj:  # Found the one with actual detail fields
                detail_data_found = data_obj
                break
        
        search_pos = detail_pos + 1
    
    if detail_data_found:
        data = detail_data_found
        
        # Define numeric fields that should be parsed to int
        numeric_fields = {"depreciation", "coe", "road_tax", "omv", "arf", 
                          "mileage", "engine_cap", "power", "dereg_value"}
        
        # Extract fields
        field_mapping = {
            "aid": "listing_id",
            "car_model": "car_model",
            "depreciation": "depreciation",
            "coe": "coe",
            "reg_date": "reg_date",
            "mileage": "mileage",
            "road_tax": "road_tax",
            "omv": "omv",
            "arf": "arf",
            "engine_cap": "engine_cap",
            "fuel_type": "fuel_type",
            "power": "power",
            "transmission": "transmission",
            "manufactured": "manufactured",
            "owners": "owners",
            "dereg_value": "dereg_value",
            "curb_weight": "curb_weight",
            "status": "status",
            "features": "features",
            "accessories": "accessories",
        }
        
        for src_field, dst_field in field_mapping.items():
            if src_field in data:
                value = data[src_field]
                # Parse dollar values for numeric fields - use None on failure
                if dst_field in numeric_fields:
                    parsed = parse_dollar_value(value)
                    detail_data[dst_field] = parsed  # None if parsing fails
                else:
                    detail_data[dst_field] = value
        
        # Extract vehicle type from nested object
        if "type_of_vehicle" in data and isinstance(data["type_of_vehicle"], dict):
            detail_data["vehicle_type"] = data["type_of_vehicle"].get("text")
    
    # Step 2: Find ucInfoDetailTopData for price
    search_pos = 0
    while True:
        top_pos = all_content.find('"ucInfoDetailTopData"', search_pos)
        if top_pos < 0:
            break
        
        success_pos = all_content.find('"success":true', top_pos)
        if success_pos > 0 and success_pos < top_pos + 100:
            top_data = find_data_object(all_content, success_pos)
            if top_data and "price" in top_data:
                detail_data["price"] = parse_dollar_value(top_data["price"])  # None if parsing fails
                break
        
        search_pos = top_pos + 1
    
    return detail_data


def coerce_dataframe_types(df: pl.DataFrame) -> pl.DataFrame:
    """Coerce DataFrame columns to consistent types.
    
    Numeric columns: Int64 (with null for unparseable values)
    String columns: String
    """
    # Define which columns should be integers
    int_columns = {
        "listing_id", "price", "depreciation", "coe", "road_tax", 
        "omv", "arf", "mileage", "engine_cap", "power", "dereg_value",
        "manufactured", "owners", "browser_seconds"
    }
    
    for col in df.columns:
        if col in int_columns:
            # Cast to Int64 (Polars handles null values correctly)
            df = df.with_columns(pl.col(col).cast(pl.Int64, strict=False))
    
    return df


# Test extraction on one HTML
if debug_html:
    test_data = extract_detail_from_html(debug_html, debug_url)
    print(f"\nTest extraction found {len(test_data)} fields:")
    for k, v in sorted(test_data.items()):
        print(f"  {k}: {v}")

# Extract from all completed jobs
print("\n" + "="*50)
print("Extracting from all completed jobs...")

all_details = []
for url, info in tqdm(job_mapping.items(), desc="Extracting data"):
    if info.get("status") == "completed" and info.get("html"):
        detail = extract_detail_from_html(info["html"], url)
        if len(detail) > 1:  # More than just the URL
            detail["job_id"] = info["job_id"]
            detail["browser_seconds"] = info.get("browser_seconds", 0)
            all_details.append(detail)

print(f"\nExtracted {len(all_details)} detail records")

# Create DataFrame
if all_details:
    detail_df = pl.DataFrame(all_details)
    
    # Coerce types to ensure consistent column types
    detail_df = coerce_dataframe_types(detail_df)
    
    print(f"\nColumns extracted: {len(detail_df.columns)}")
    print(f"Columns: {sorted(detail_df.columns)}")
    
    # Show column types
    print(f"\nColumn types:")
    for col in sorted(detail_df.columns):
        print(f"  {col}: {detail_df[col].dtype}")
    
    # Show field completeness
    print(f"\nField completeness:")
    for col in sorted(detail_df.columns):
        non_null = detail_df.height - detail_df[col].null_count()
        pct = (non_null / detail_df.height) * 100
        print(f"  {col}: {non_null}/{detail_df.height} ({pct:.0f}%)")
    
    print(f"\nSample data (key columns):")
    key_cols = ["listing_id", "car_model", "price", "depreciation", "omv", "arf", "coe", "mileage"]
    existing_cols = [c for c in key_cols if c in detail_df.columns]
    print(detail_df.select(existing_cols).head(5))
    
    detail_df.write_parquet("output/validation_detail_data.parquet", compression="snappy")
    print("\n✅ Saved to output/validation_detail_data.parquet")
else:
    print("⚠️ No data extracted - check extraction logic")
    print("   Saving sample HTML for manual inspection...")
    if debug_html:
        with open("output/sample_detail_page.html", "w") as f:
            f.write(debug_html)
        print("   ✅ Saved sample HTML to output/sample_detail_page.html")

## 7. Data Quality Checks (AC5)

Validate success rate, field completeness, and data integrity.

In [7]:
print("\n=== AC5: Data Quality Checks ===\n")

if all_details and len(all_details) > 0:
    # Calculate success rate
    success_rate = (completed_count / len(sample_urls)) * 100
    print(f"Success Rate: {success_rate:.1f}% (target: >90%)")
    success_pass = success_rate >= 90
    
    # Check listing_id field
    if 'listing_id' in detail_df.columns:
        null_ids = detail_df.filter(pl.col('listing_id').is_null()).height
        print(f"\nNull listing_ids: {null_ids} (target: 0)")
        null_id_pass = null_ids == 0
    else:
        print("\n⚠️ 'listing_id' field not found in extracted data")
        null_id_pass = False
    
    # Check detail_page_url completeness
    if 'detail_page_url' in detail_df.columns:
        null_urls = detail_df.filter(pl.col('detail_page_url').is_null()).height
        print(f"Null detail_page_url: {null_urls} (target: 0)")
        null_url_pass = null_urls == 0
    else:
        null_url_pass = False
    
    # Key field completeness
    key_fields = ['price', 'depreciation', 'omv', 'arf', 'coe', 'road_tax', 'mileage', 'engine_cap']
    print(f"\nKey Field Completeness (target: >90%):")
    
    completeness_pass = True
    for field in key_fields:
        if field in detail_df.columns:
            non_null = detail_df.height - detail_df[field].null_count()
            non_null_pct = (non_null / detail_df.height) * 100
            status = "✅" if non_null_pct >= 90 else "⚠️"
            print(f"  {status} {field:15s} {non_null}/{detail_df.height} ({non_null_pct:.0f}%)")
            if non_null_pct < 90:
                completeness_pass = False
        else:
            print(f"  ❌ {field:15s} NOT FOUND")
            completeness_pass = False
    
    # Overall assessment
    print("\n" + "="*50)
    all_pass = success_pass and null_id_pass and null_url_pass and completeness_pass
    if all_pass:
        print("✅ ALL AC5 CRITERIA MET - Ready for production")
    else:
        print("⚠️ AC5 criteria not fully met - review issues above")
        if not completeness_pass:
            print("   Note: Some fields may have different formats or be missing from certain listings")
else:
    print("❌ No data to validate - extraction failed")
    print("   This could be due to:")
    print("   - Crawl jobs not completed")
    print("   - RSC payload structure changed")
    print("   - Extraction logic needs adjustment")


=== AC5: Data Quality Checks ===

Success Rate: 100.0% (target: >90%)

Null listing_ids: 0 (target: 0)
Null detail_page_url: 0 (target: 0)

Key Field Completeness (target: >90%):
  ✅ price           20/20 (100%)
  ✅ depreciation    20/20 (100%)
  ✅ omv             20/20 (100%)
  ✅ arf             20/20 (100%)
  ✅ coe             20/20 (100%)
  ✅ road_tax        20/20 (100%)
  ✅ mileage         20/20 (100%)
  ✅ engine_cap      20/20 (100%)

✅ ALL AC5 CRITERIA MET - Ready for production


## 8. Cost Estimation for Full Run (AC6)

Calculate estimated cost for scraping all detail pages (count derived from actual data).

In [ ]:
print("\n=== AC6: Cost Estimation for Full Production Run ===\n")

# Calculate metrics from validation
completed = sum(1 for v in job_mapping.values() if v.get("status") == "completed")
total_browser_seconds = sum(
    v.get("browser_seconds", 0) 
    for v in job_mapping.values() 
    if v.get("status") == "completed"
)

avg_browser_seconds = total_browser_seconds / completed if completed > 0 else 6.0
success_rate = (completed / len(job_mapping)) * 100 if completed > 0 else 0.0

# Use actual URL count from loaded data (not hardcoded)
# Falls back to 13800 estimate if TOTAL_DETAIL_URLS_AVAILABLE is not defined
total_detail_urls = TOTAL_DETAIL_URLS_AVAILABLE if 'TOTAL_DETAIL_URLS_AVAILABLE' in dir() else 13800

est_total_hours = (total_detail_urls * avg_browser_seconds) / 3600
est_cost = 5 + max(0, est_total_hours - 10) * 0.09

print(f"Validation Results:")
print(f"  Sample size: {len(sample_urls)}")
print(f"  Completed: {completed}/{len(sample_urls)}")
print(f"  Avg browser time per page: {avg_browser_seconds:.2f}s")
print(f"  Success rate: {success_rate:.1f}%")

print(f"\nFull Production Estimate:")
print(f"  Total detail URLs: {total_detail_urls:,}")
print(f"  Estimated browser hours: {est_total_hours:.1f}h")
print(f"  Free tier: 10h")
print(f"  Overage: {max(0, est_total_hours - 10):.1f}h @ $0.09/h")
print(f"  Monthly plan: $5.00")
print(f"  Estimated total cost: ${est_cost:.2f}")

## 9. Save Validation Report (AC6)

In [ ]:
print("\n=== Saving Validation Report ===\n")

# Calculate summary stats
completed_count = sum(1 for v in job_mapping.values() if v.get("status") == "completed")
success_rate_val = (completed_count / len(job_mapping)) * 100 if len(job_mapping) > 0 else 0.0
fields_extracted = len(detail_df.columns) if 'detail_df' in dir() and detail_df is not None else 0

# Get total URLs from actual data (not hardcoded)
total_urls_for_report = total_detail_urls if 'total_detail_urls' in dir() else (
    TOTAL_DETAIL_URLS_AVAILABLE if 'TOTAL_DETAIL_URLS_AVAILABLE' in dir() else 0
)

# Build report
validation_report = {
    "timestamp": datetime.now().isoformat(),
    "sample_size": len(sample_urls),
    "completed_jobs": completed_count,
    "success_rate": success_rate_val,
    "fields_extracted": fields_extracted,
    "records_extracted": len(all_details) if 'all_details' in dir() else 0,
    "cost_estimate": {
        "avg_browser_seconds": avg_browser_seconds if 'avg_browser_seconds' in dir() else 0,
        "total_urls": total_urls_for_report,
        "estimated_hours": est_total_hours if 'est_total_hours' in dir() else 0,
        "estimated_cost": est_cost if 'est_cost' in dir() else 0
    },
    "ac5_passed": all_pass if 'all_pass' in dir() else False,
}

with open("output/validation_report.json", 'w') as f:
    json.dump(validation_report, f, indent=2)

print("✅ Validation report saved to output/validation_report.json")
print(f"\nSummary:")
print(f"  Completed jobs: {completed_count}/{len(sample_urls)}")
print(f"  Success rate: {success_rate_val:.1f}%")
print(f"  Records extracted: {len(all_details) if 'all_details' in dir() else 0}")
print(f"  Fields extracted: {fields_extracted}")

## 10. Summary & Next Steps (AC7)

In [10]:
print("\n=== AC7: Summary & Recommendations ===\n")

# Calculate summary stats safely
completed_count = sum(1 for v in job_mapping.values() if v.get("status") == "completed")
success_rate_val = (completed_count / len(job_mapping)) * 100 if len(job_mapping) > 0 else 0.0
fields_count = len(detail_df.columns) if 'detail_df' in dir() else 0
records_count = len(all_details) if 'all_details' in dir() else 0

print("VALIDATION RESULTS:")
print(f"  ✅ Sample crawl completed: {completed_count}/{len(sample_urls)} jobs")
print(f"  ✅ Data extraction: {fields_count} fields from {records_count} records")
print(f"  ✅ Success rate: {success_rate_val:.1f}%")

print("\nRECOMMENDATIONS:")

if success_rate_val >= 90 and records_count > 0:
    print("  ✅ APPROVED: Crawl-only approach is viable")
    print("  → Proceed to production implementation")
    est_hours = est_total_hours if 'est_total_hours' in dir() else 0
    est_cost_val = est_cost if 'est_cost' in dir() else 0
    print(f"  → Estimated time: {est_hours:.1f} hours (~{est_hours/24:.1f} days)")
    print(f"  → Estimated cost: ${est_cost_val:.2f}")
elif success_rate_val >= 90:
    print("  ⚠️ PARTIAL SUCCESS: Crawling works but extraction needs improvement")
    print("  → Review RSC extraction logic")
    print("  → Check HTML structure of detail pages")
else:
    print("  ⚠️ REVIEW NEEDED: Some criteria not met")
    print("  → Investigate failed jobs")
    print("  → Review RSC extraction logic")
    print("  → Re-run validation after fixes")

print("\nNEXT STEPS:")
print("  1. Document findings in GitHub issue #3 comments")
print("  2. If approved, create production implementation notebook")
print("  3. If issues found, debug and re-validate")

print("\n✅ US-3 VALIDATION COMPLETE")


=== AC7: Summary & Recommendations ===

VALIDATION RESULTS:
  ✅ Sample crawl completed: 20/20 jobs
  ✅ Data extraction: 25 fields from 20 records
  ✅ Success rate: 100.0%

RECOMMENDATIONS:
  ✅ APPROVED: Crawl-only approach is viable
  → Proceed to production implementation
  → Estimated time: 2.3 hours (~0.1 days)
  → Estimated cost: $5.00

NEXT STEPS:
  1. Document findings in GitHub issue #3 comments
  2. If approved, create production implementation notebook
  3. If issues found, debug and re-validate

✅ US-3 VALIDATION COMPLETE


---

## Acceptance Criteria Summary

| AC | Description | Status |
|----|-------------|--------|
| AC1 | Load sample detail URLs | ⬜ |
| AC2 | Submit crawl jobs with correct params | ⬜ |
| AC3 | Poll and collect results | ⬜ |
| AC4 | Extract detail fields from RSC payload | ⬜ |
| AC5 | Data quality checks | ⬜ |
| AC6 | Cost estimation report | ⬜ |
| AC7 | Document findings | ⬜ |

---

## 11. Data Analysis

View the full extracted dataset.

In [11]:
print("\n=== Data Analysis: Full Dataset View ===\n")

# Read the parquet file
df = pl.read_parquet("output/validation_detail_data.parquet")

print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"\nColumns: {sorted(df.columns)}")

# Display full dataframe with all columns
print("\n" + "="*80)
print("FULL DATASET")
print("="*80)

# Configure polars to show all columns and full string content
pl.Config(set_fmt_str_lengths=100)
pl.Config(set_fmt_table_cell_list_len=10)

# Show all rows with selected key columns first
key_cols = [
    "listing_id", "car_model", "price", "depreciation", 
    "omv", "arf", "coe", "road_tax", "mileage", 
    "engine_cap", "fuel_type", "transmission", "vehicle_type", "owners"
]
existing_key_cols = [c for c in key_cols if c in df.columns]

print(f"\nKey Fields ({len(existing_key_cols)} columns):")
print(df.select(existing_key_cols))

# Summary statistics for numeric columns
print("\n" + "="*80)
print("SUMMARY STATISTICS (Numeric Fields)")
print("="*80)

numeric_cols = ["price", "depreciation", "omv", "arf", "coe", "road_tax", "mileage", "engine_cap", "power"]
existing_numeric = [c for c in numeric_cols if c in df.columns and df[c].dtype in [pl.Int64, pl.Float64]]

if existing_numeric:
    print(df.select(existing_numeric).describe())

# Export to CSV for easy viewing
df.write_csv("output/validation_detail_data.csv")
print(f"\n✅ Also exported to CSV: output/validation_detail_data.csv")


=== Data Analysis: Full Dataset View ===

Dataset Shape: 20 rows × 25 columns

Columns: ['accessories', 'arf', 'browser_seconds', 'car_model', 'coe', 'curb_weight', 'depreciation', 'dereg_value', 'detail_page_url', 'engine_cap', 'features', 'fuel_type', 'job_id', 'listing_id', 'manufactured', 'mileage', 'omv', 'owners', 'power', 'price', 'reg_date', 'road_tax', 'status', 'transmission', 'vehicle_type']

FULL DATASET

Key Fields (14 columns):
shape: (20, 14)
┌────────────┬────────────┬────────┬────────────┬───┬────────────┬────────────┬───────────┬────────┐
│ listing_id ┆ car_model  ┆ price  ┆ depreciati ┆ … ┆ fuel_type  ┆ transmissi ┆ vehicle_t ┆ owners │
│ ---        ┆ ---        ┆ ---    ┆ on         ┆   ┆ ---        ┆ on         ┆ ype       ┆ ---    │
│ i64        ┆ str        ┆ i64    ┆ ---        ┆   ┆ str        ┆ ---        ┆ ---       ┆ str    │
│            ┆            ┆        ┆ str        ┆   ┆            ┆ str        ┆ str       ┆        │
╞════════════╪════════════╪═════